STATISTICAL OUTLIER DETECTION
___

BUSINESS QUESTION

1. Are there layoff events that are statistically unusual RELATIVE TO what's normal for a company of that funding size - not just "high percentage" in an absolute sense?
2. A 40% layoff is very different for a Mega-funded company than for a Small startup, so we compare each event to its own funding bracket's typical range, not to the whole dataset at once.

In [2]:
import pandas as pd

Load the master dataset

In [3]:
df = pd.read_csv("D:\DATA ANALYSIS\MySQL data project(A)\Data\Processed\layoffs_master_enriched(1).csv", parse_dates=["date"])

Calculating the IQR boundaries for EACH funding bracket

In [4]:
q1_by_bracket = df.groupby("funding_bracket")["percentage_laid_off"].transform(
    lambda x: x.quantile(0.25)
)

q3_by_bracket = df.groupby("funding_bracket")["percentage_laid_off"].transform(
    lambda x: x.quantile(0.75)
)

iqr_by_bracket = q3_by_bracket - q1_by_bracket

lower_bound = q1_by_bracket - 1.5 * iqr_by_bracket
upper_bound = q3_by_bracket + 1.5 * iqr_by_bracket

Flaging outliers and label their direction

In [5]:
df["is_statistical_outlier"] = (
    (df["percentage_laid_off"] < lower_bound) | (df["percentage_laid_off"] > upper_bound)
)

import numpy as np

conditions = [
    df["percentage_laid_off"] > upper_bound,
    df["percentage_laid_off"] < lower_bound,
]

choices = ["Unusually High for Funding Level", "Unusually Low for Funding Level"]

df["outlier_direction"] = np.select(conditions, choices, default="Typical")

Review the results

In [6]:
print("Outlier count by direction:")
print(df["outlier_direction"].value_counts())

print("\nOutlier rate by funding bracket:")

outlier_rate_by_bracket = df.groupby("funding_bracket", observed=True)[
    "is_statistical_outlier"
].mean()

print((outlier_rate_by_bracket * 100).round(1))

print("\nSample of flagged 'Unusually High' events:")

high_outliers = df[df["outlier_direction"] == "Unusually High for Funding Level"]

print(
    high_outliers[
        ["company", "funding_bracket", "percentage_laid_off", "total_laid_off"]
    ]
    .sort_values(by="percentage_laid_off", ascending=False)
    .head(8)
    .to_string(index=False)
)

Outlier count by direction:
outlier_direction
Typical                             1900
Unusually High for Funding Level      94
Name: count, dtype: int64

Outlier rate by funding bracket:
funding_bracket
Large ($500M-2B)     5.4
Medium ($50-500M)    7.5
Mega (>$2B)          4.4
Small (<$50M)        0.0
Name: is_statistical_outlier, dtype: float64

Sample of flagged 'Unusually High' events:
            company   funding_bracket  percentage_laid_off  total_laid_off
            Airlift Medium ($50-500M)                  1.0               0
Deliveroo Australia  Large ($500M-2B)                  1.0             120
            Katerra  Large ($500M-2B)                  1.0            2434
    Motif Investing Medium ($50-500M)                  1.0               0
            Openpay Medium ($50-500M)                  1.0              83
              Hubba Medium ($50-500M)                  1.0              45
               HOOQ Medium ($50-500M)                  1.0             250
       

Save the final enriched master dataset

In [7]:
df.to_csv("layoffs_master_enriched.csv", index=False)

print("\nUpdated: layoffs_master_enriched.csv")
print(f"Final columns in the master dataset: {list(df.columns)}")


Updated: layoffs_master_enriched.csv
Final columns in the master dataset: ['company', 'location', 'industry', 'total_laid_off', 'percentage_laid_off', 'date', 'stage', 'country', 'funds_raised_millions', 'number_of_layoff_events', 'is_repeat_layoff_company', 'log_funds_raised', 'funding_bracket', 'severity_tier', 'is_statistical_outlier', 'outlier_direction']
